# 03 homa packets

Question: does the compact wire form of the crate `homa` write and read every packet type as the
Homa design note, first version, section 2.3 states, and refuse every malformed packet with the
error the decoding rules name.

The side under test is `aosabi-devtool homa-packet`, which encodes and decodes one packet through
`homa::wire::Compact` per request line. The reference side is `tests/harness/homa/compact.py`,
written from the same section of the design note and sharing no code with the crate.

Four sections run. The eight packet types of the form go through both sides in both directions,
1,000 seeded valid packets do the same, 64 hostile packets are refused with the error each one is
written to meet, and a seeded run of damaged and random byte strings goes through both decoders,
whose answers must agree. A DATA packet carries seeded bytes and Japanese text in UTF-8 as plain
bytes, because the Homa layer carries bytes and the framework is meant to serve Japanese content.

The exit number is that of stage H0 of the plan's section The notebooks: 0 byte differences over
the 8 types and the 1,000 seeded valid packets, 64 of 64 hostile packets refused with the expected
error, and 0 panics.

Judged stage: H0 of the Homa track.

In [ ]:
# Papermill overrides this cell.
# The tree under comparison. Papermill is invoked from the repository root,
# and the notebook walks up from the working directory when it is invoked from
# elsewhere.
aosabi_root = "."

# The seed of the valid packets and of the damaged byte strings. Another seed
# draws another corpus of the same shape.
seed = 20260927

# The valid packets of the exit number.
max_ops = 1000

# The damaged and random byte strings both decoders read.
fuzz_ops = 10000

# The per command timeout.
timeout_s = 600

# The pinned toolchain the crate is built with.
rust_version = "1.95.0"

## Setup

The helpers come from `tests/harness`, and the reference from `tests/harness/homa`. The cell prints
the toolchain, the installed targets and the gates of the plan's section The commands that the
notebook rests on: the build of the devtool, the lints of the workspace and the format of the crates
`homa` and `aosabi-devtool`. Every cargo call passes `--locked`, so a lock file that does not match
the manifests fails the run instead of moving. Every batch is written under
`tests/notebooks/out/03_homa_packets/`, which git ignores, so a failing case is replayed by hand
with the command the notebook runs.

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import bytediff, paths, run
from harness.homa import compact

P = paths(aosabi_root)
OUT = P.out / "03_homa_packets"
OUT.mkdir(parents=True, exist_ok=True)
DEVTOOL = P.target / "debug" / "aosabi-devtool"
CARGO_BIN = Path(os.environ.get("CARGO_HOME") or Path.home() / ".cargo") / "bin"
CARGO = CARGO_BIN / "cargo"
RUSTUP = CARGO_BIN / "rustup"
RUSTFMT = CARGO_BIN / "rustfmt"

# The gates of the plan's section The commands that this notebook rests on,
# each cargo call with --locked: the build of the seam it drives, the lints of
# the workspace, and the format of the crate homa under its own edition 2021
# and of the devtool, which rustfmt checks because cargo fmt takes no
# --locked.
GATES = [
    ("cargo build --locked -p aosabi-devtool",
     [CARGO, "build", "--locked", "-p", "aosabi-devtool"]),
    ("cargo clippy --locked --workspace --all-targets -- -D warnings",
     [CARGO, "clippy", "--locked", "--workspace", "--all-targets", "--", "-D", "warnings"]),
    ("rustfmt --check --edition 2021 crates/homa/src/lib.rs",
     [RUSTFMT, "--check", "--edition", "2021", P.root / "crates" / "homa" / "src" / "lib.rs"]),
    ("rustfmt --check --edition 2024 crates/aosabi-devtool/src/main.rs",
     [RUSTFMT, "--check", "--edition", "2024", P.root / "crates" / "aosabi-devtool" / "src" / "main.rs"]),
]
TARGETS = run([RUSTUP, "target", "list", "--installed"], cwd=P.root, timeout_s=60).out.split()
GATE_FAILURES = []
for label, argv in GATES:
    gate = run(argv, cwd=P.root, timeout_s=900, check=False)
    print("gate %-66s exit %d, %.1f s" % (label, gate.returncode, gate.duration_s))
    if gate.returncode != 0:
        GATE_FAILURES.append(label)
        print((gate.err or gate.out).strip()[-2000:])

reported_cargo = run([CARGO, "--version"], cwd=P.root, timeout_s=timeout_s).out.strip()
smoke = json.loads(run([DEVTOOL, "homa-packet", "--smoke"], cwd=P.root, timeout_s=timeout_s).out)
print("repository root:", P.rel(P.root))
print("devtool        :", P.rel(DEVTOOL))
print("batches        :", P.rel(OUT))
print("cargo          :", reported_cargo)
print("targets        :", ", ".join(TARGETS))
print("seam           :", json.dumps(smoke))
print("seed           :", seed)
assert not GATE_FAILURES, GATE_FAILURES
assert rust_version in reported_cargo, "the pinned toolchain is %s, this is %s" % (
    rust_version,
    reported_cargo,
)
assert smoke == {"subcommand": "homa-packet", "implemented": True, "lands_at": "H0"}, smoke

PANICS = []
DIFFS = {}


def batch(requests, name):
    """Send `requests` to `homa-packet` in one process and return its answers.

    A process that exits other than 0 or answers fewer lines than it read
    is a panic or a crash of the seam, which the verdict counts.
    """
    path = OUT / ("%s.ndjson" % name)
    path.write_text("".join(json.dumps(r) + "\n" for r in requests), encoding="utf-8")
    result = run(
        [DEVTOOL, "homa-packet", "--requests", path], cwd=P.root, timeout_s=timeout_s, check=False
    )
    answers = [json.loads(line) for line in result.out.splitlines()]
    if result.returncode != 0 or len(answers) != len(requests):
        PANICS.append("%s: exit %d, %d answers to %d requests, %s" % (
            name, result.returncode, len(answers), len(requests), result.err.strip()[-300:]))
    fatal = [a for a in answers if "fatal" in a]
    assert not fatal, "the seam refused requests of %s: %s" % (name, fatal[:3])
    return answers


def python_encode(packet, need_limits):
    try:
        return compact.encode(packet, need_limits).hex()
    except compact.NoRoom:
        return "NoRoom"


def python_decode(hex_bytes):
    try:
        return {"decoded": compact.decode(bytes.fromhex(hex_bytes))}
    except compact.DecodeError as error:
        return {"error": error.name}


def rust_value(answer, key):
    return answer.get(key, answer.get("error"))


def both_ways(name, packets):
    """Encode `packets` on both sides, decode each side's bytes on the other,
    and compare the streams byte for byte."""
    requests = [
        {"id": i, "op": "encode", "form": "compact", "packet": p, "need_limits": n}
        for i, (p, n) in enumerate(packets)
    ]
    encoded = batch(requests, name + "-encode")
    rust = [rust_value(a, "hex") for a in encoded]
    python = [python_encode(p, n) for p, n in packets]
    differing = [i for i, (r, q) in enumerate(zip(rust, python)) if r != q]
    decode_rust = batch(
        [{"id": i, "op": "decode", "form": "compact", "hex": h} for i, h in enumerate(python)],
        name + "-decode",
    )
    wrong_rust = [
        i for i, (a, (p, n)) in enumerate(zip(decode_rust, packets))
        if a.get("decoded") != {"ports": {"src": 0, "dst": 0}, "packet": p, "need_limits": n}
    ]
    wrong_python = [
        i for i, (h, (p, n)) in enumerate(zip(rust, packets))
        if python_decode(h) != {"decoded": {"ports": {"src": 0, "dst": 0}, "packet": p, "need_limits": n}}
    ]
    DIFFS[name] = len(differing) + len(wrong_rust) + len(wrong_python)
    print("%s: %d packets, %d encodings differ, %d packets the crate reads back wrong,"
          " %d the reference reads back wrong" % (
              name, len(packets), len(differing), len(wrong_rust), len(wrong_python)))
    for i in (differing + wrong_rust + wrong_python)[:3]:
        print("  case", i, json.dumps(packets[i][0])[:160])
        print("    crate    ", rust[i][:120], json.dumps(decode_rust[i])[:160])
        print("    reference", python[i][:120])
    bytediff(
        "\n".join(python).encode(), "\n".join(rust).encode(), name + " encodings, reference against crate"
    )
    return rust

## The eight packet types

One packet of each type the compact form carries, DATA as a whole message and as a part, GRANT,
RESEND, `RPC_UNKNOWN`, BUSY, `NEED_ACK`, ACK and LIMITS, each with and without the flag
`NEED_LIMITS`.

In [ ]:
JAPANESE = [
    "日本語のテキスト",
    "東京都渋谷区神南一丁目",
    "こんにちは、世界。",
    "漢字とひらがなとカタカナ",
    "{\"data\":{\"名前\":\"山田太郎\"}}",
]


def data_packet(ident, payload, msg_length=None, offset=0, ack=None, retransmit=False):
    return {
        "type": "DATA",
        "id": ident,
        "msg_length": len(payload) if msg_length is None else msg_length,
        "offset": offset,
        "retransmit": retransmit,
        "ack": None if ack is None else {"client_id": ack, "server_port": 0},
        "cutoff_version": 0,
        "incast": False,
        "payload": payload.hex(),
    }


TYPES = [
    ("DATA whole", data_packet(0x1000_0002, JAPANESE[0].encode())),
    ("DATA part", data_packet(0x1000_0003, JAPANESE[1].encode(), msg_length=40_000, offset=1_680)),
    ("DATA part with ACK", data_packet(0x1000_0003, b"\x00\x01\x02", msg_length=9, offset=6, ack=0x1000_0000, retransmit=True)),
    ("GRANT", {"type": "GRANT", "id": 0x1000_0003, "offset": 4_096, "priority": 7}),
    ("RESEND", {"type": "RESEND", "id": 0x1000_0002, "offset": 0, "length": 0xFFFF_FFFF, "priority": 7}),
    ("RPC_UNKNOWN", {"type": "RPC_UNKNOWN", "id": 0x1000_0003}),
    ("BUSY", {"type": "BUSY", "id": 0x1000_0003}),
    ("NEED_ACK", {"type": "NEED_ACK", "id": 0x1000_0003}),
] + [
    ("ACK of %d" % n, {"type": "ACK", "id": 0, "acks": [{"client_id": 2 * k + 2, "server_port": 0} for k in range(n)]})
    for n in range(1, 6)
] + [
    ("LIMITS", {"type": "LIMITS", "max_packet": 256, "max_message": 65_536, "unsched_bytes": 512, "max_rpcs": 8}),
]

type_packets = [(p, n) for _, p in TYPES for n in (False, True)]
type_rust = both_ways("types", type_packets)
print()
print("%-20s %-12s %s" % ("packet", "header bytes", "crate bytes with NEED_LIMITS off"))
HEADER_BYTES = {}
for (label, packet), hexed in zip(TYPES, type_rust[0::2]):
    payload = len(bytes.fromhex(packet.get("payload", ""))) if packet["type"] == "DATA" else 0
    HEADER_BYTES[label] = len(bytes.fromhex(hexed)) - payload
    print("%-20s %-12d %s" % (label, HEADER_BYTES[label], hexed[:60]))
kinds = sorted({p["type"] for _, p in TYPES})
print()
print("types covered:", len(kinds), kinds)
assert len(kinds) == 8, kinds

## 1,000 seeded valid packets

The generator draws the type, the identifier of 32 bits, the flags and every field inside its
range, and a DATA packet carries seeded bytes or a run of Japanese text, whole or as a part whose
offset and payload stay inside its length. Each packet goes through both sides in both
directions, and the streams of both encoders are compared byte for byte.

In [ ]:
def random_payload(g):
    if g.random() < 0.3:
        text = g.choice(JAPANESE).encode() * g.randint(1, 8)
        return text[: g.randint(0, len(text))] if g.random() < 0.5 else text
    return bytes(g.getrandbits(8) for _ in range(g.randint(0, 300)))


def random_packet(g):
    kind = g.choice(["DATA", "DATA", "DATA", "GRANT", "RESEND", "RPC_UNKNOWN", "BUSY", "NEED_ACK", "ACK", "LIMITS"])
    ident = g.getrandbits(32)
    if kind == "DATA":
        payload = random_payload(g)
        ack = g.getrandbits(32) if g.random() < 0.3 else None
        retransmit = g.random() < 0.2
        if g.random() < 0.4 or not payload:
            return data_packet(ident, payload, ack=ack, retransmit=retransmit)
        msg_length = g.randint(len(payload) + 1, len(payload) + 1_000_000)
        offset = g.randint(0, msg_length - len(payload))
        return data_packet(ident, payload, msg_length=msg_length, offset=offset, ack=ack, retransmit=retransmit)
    if kind == "GRANT":
        return {"type": "GRANT", "id": ident, "offset": g.getrandbits(32), "priority": 7}
    if kind == "RESEND":
        return {"type": "RESEND", "id": ident, "offset": g.getrandbits(32), "length": g.getrandbits(32), "priority": 7}
    if kind == "ACK":
        return {"type": "ACK", "id": ident, "acks": [{"client_id": g.getrandbits(32), "server_port": 0} for _ in range(g.randint(1, 5))]}
    if kind == "LIMITS":
        return {
            "type": "LIMITS",
            "max_packet": g.randint(20, 0xFFFF),
            "max_message": g.randint(1, 0xFFFF_FFFF),
            "unsched_bytes": g.randint(1, 0xFFFF_FFFF),
            "max_rpcs": g.randint(1, 255),
        }
    return {"type": kind, "id": ident}


generator = random.Random(seed)
valid = [(random_packet(generator), generator.random() < 0.5) for _ in range(max_ops)]
by_type = {}
for packet, _ in valid:
    by_type[packet["type"]] = by_type.get(packet["type"], 0) + 1
japanese = sum(
    1 for p, _ in valid
    if p["type"] == "DATA" and any(t.encode()[:6] in bytes.fromhex(p["payload"]) for t in JAPANESE)
)
print("valid packets:", len(valid), "by type:", by_type)
print("DATA packets that carry Japanese text:", japanese)
valid_rust = both_ways("valid", valid)

## 64 hostile packets

Each hostile packet is written to break one rule of the decoder, and the table names the error the
rules of reading 5 give it. The crate's answer and the reference's answer must both equal that
name. The list holds truncated packets, set reserved bits, other versions, lengths off by one, an
ACK of 6 bytes and one of 30, a part of DATA whose offset runs past its length, a LIMITS of 17
bytes and one with `max_packet` 19, the codes 0x15, 0x16 and 0x19 of the full form, codes Homa
does not have, flags only DATA may set on other types, and the order in which two broken rules
are named.

In [ ]:
def common(kind, flags, ident):
    return bytes([kind, flags]) + ident.to_bytes(4, "big")


part = common(0x10, 0, 2) + (100).to_bytes(4, "big") + (10).to_bytes(4, "big") + b"\x01\x02\x03"
limits = common(0x20, 0, 0) + bytes([8, 0]) + (256).to_bytes(2, "big") + (512).to_bytes(4, "big") + (65_536).to_bytes(4, "big")


def limits_with(**changes):
    body = bytearray(limits)
    if "max_rpcs" in changes:
        body[6] = changes["max_rpcs"]
    if "reserved" in changes:
        body[7] = changes["reserved"]
    if "max_packet" in changes:
        body[8:10] = changes["max_packet"].to_bytes(2, "big")
    if "unsched" in changes:
        body[10:14] = changes["unsched"].to_bytes(4, "big")
    if "max_message" in changes:
        body[14:18] = changes["max_message"].to_bytes(4, "big")
    if "sender" in changes:
        body[2:6] = changes["sender"].to_bytes(4, "big")
    return bytes(body)


HOSTILE = []
for n in range(6):
    HOSTILE.append(("truncated to %d bytes" % n, part[:n], "Short"))
for n in range(6, 14):
    HOSTILE.append(("part of DATA truncated to %d bytes" % n, part[:n], "Length"))
for flags, label in ((0x10, "bit 4"), (0x20, "bit 5"), (0x30, "bits 4 and 5")):
    HOSTILE.append(("DATA with reserved %s" % label, bytes([0x10, flags]) + part[2:], "Reserved"))
HOSTILE.append(("GRANT with reserved bit 4", common(0x11, 0x10, 3) + (5).to_bytes(4, "big"), "Reserved"))
for version in (1, 2, 3):
    HOSTILE.append(("version %d" % version, bytes([0x10, version << 6]) + part[2:], "Version"))
for kind, length, label in (
    (0x11, 9, "GRANT"), (0x11, 11, "GRANT"), (0x12, 13, "RESEND"), (0x12, 15, "RESEND"),
    (0x13, 7, "RPC_UNKNOWN"), (0x14, 7, "BUSY"), (0x17, 7, "NEED_ACK"),
    (0x18, 9, "ACK"), (0x18, 11, "ACK"), (0x11, 6, "GRANT"), (0x12, 6, "RESEND"),
):
    HOSTILE.append(("%s of %d bytes" % (label, length), common(kind, 0, 3) + bytes(length - 6), "Length"))
HOSTILE.append(("ACK of 6 bytes", common(0x18, 0, 0), "Length"))
HOSTILE.append(("ACK of 30 bytes, six entries", common(0x18, 0, 0) + bytes(24), "Length"))
HOSTILE.append(("LIMITS of 17 bytes", limits[:17], "Length"))
HOSTILE.append(("LIMITS of 19 bytes", limits + b"\x00", "Length"))
HOSTILE.append(("LIMITS with max_packet 19", limits_with(max_packet=19), "Range"))
HOSTILE.append(("LIMITS with max_rpcs 0", limits_with(max_rpcs=0), "Range"))
HOSTILE.append(("LIMITS with unsched_bytes 0", limits_with(unsched=0), "Range"))
HOSTILE.append(("LIMITS with max_message 0", limits_with(max_message=0), "Range"))
HOSTILE.append(("LIMITS with its reserved byte set", limits_with(reserved=1), "Reserved"))
HOSTILE.append(("LIMITS that names a sender", limits_with(sender=7), "Reserved"))
HOSTILE.append(("part of DATA whose offset runs past its length", common(0x10, 0, 2) + (12).to_bytes(4, "big") + (10).to_bytes(4, "big") + b"\x01\x02\x03", "Range"))
HOSTILE.append(("part of DATA whose end passes 2^32", common(0x10, 0, 2) + (0xFFFF_FFFF).to_bytes(4, "big") + (0xFFFF_FFFE).to_bytes(4, "big") + b"\x01\x02\x03", "Range"))
HOSTILE.append(("part of DATA of msg_length 0", common(0x10, 0, 2) + bytes(8) + b"\x01", "Range"))
HOSTILE.append(("part of DATA without payload", part[:14], "Length"))
HOSTILE.append(("whole DATA with ACK and 9 bytes", common(0x10, 0x06, 2) + b"\x00\x00\x00", "Length"))
HOSTILE.append(("part of DATA with ACK and 17 bytes", common(0x10, 0x02, 2) + (100).to_bytes(4, "big") + (10).to_bytes(4, "big") + b"\x00\x00\x00", "Length"))
for kind, label in ((0x15, "CUTOFFS"), (0x16, "FREEZE"), (0x19, "START_MSG")):
    HOSTILE.append(("%s, of the full form" % label, common(kind, 0, 3) + bytes(20), "Form"))
for kind in (0x00, 0x0F, 0x1A, 0x21, 0xFF):
    HOSTILE.append(("type 0x%02x" % kind, common(kind, 0, 3) + bytes(8), "Type"))
HOSTILE.append(("GRANT with RETRANSMIT", common(0x11, 0x01, 3) + (5).to_bytes(4, "big"), "Reserved"))
HOSTILE.append(("BUSY with ACK", common(0x14, 0x02, 3), "Reserved"))
HOSTILE.append(("LIMITS with WHOLE", bytes([0x20, 0x04]) + limits[2:], "Reserved"))
HOSTILE.append(("NEED_ACK with WHOLE", common(0x17, 0x04, 3), "Reserved"))
HOSTILE.append(("version 1 with reserved bits, version first", bytes([0x10, 0x70]) + part[2:], "Version"))
HOSTILE.append(("unknown type with version 2, version first", bytes([0x42, 0x80]) + bytes(4), "Version"))
HOSTILE.append(("unknown type with reserved bit 5, reserved first", bytes([0x42, 0x20]) + bytes(4), "Reserved"))
HOSTILE.append(("FREEZE with RETRANSMIT, form first", common(0x16, 0x01, 3), "Form"))

print("hostile packets:", len(HOSTILE))
assert len(HOSTILE) == 64, len(HOSTILE)
answers = batch(
    [{"id": i, "op": "decode", "form": "compact", "hex": h.hex()} for i, (_, h, _) in enumerate(HOSTILE)],
    "hostile",
)
refused = 0
W = (54, 10, 10, 10)
print("%-54s %-10s %-10s %-10s" % ("packet", "expected", "crate", "reference"))
for (label, data, expected), answer in zip(HOSTILE, answers):
    crate = answer.get("error", "decoded")
    reference = python_decode(data.hex()).get("error", "decoded")
    ok = crate == expected == reference
    refused += ok
    print("%-54s %-10s %-10s %-10s%s" % (label[:54], expected, crate, reference, "" if ok else "  DIFFERS"))
print()
print("hostile packets refused with the expected error: %d of %d" % (refused, len(HOSTILE)))

## Damaged and random byte strings

The seeded run takes the encodings of the valid packets and damages each one, a byte flipped,
cut, dropped, duplicated or appended, and adds strings of random bytes of 0 to 40 bytes. Both
decoders read every string, and for each one they must both decode the same packet or both name
the same error. The seam must answer every line, which is the count of panics the exit number
asks for.

In [ ]:
fuzz = random.Random(seed ^ 0x03)
strings = []
for n in range(fuzz_ops):
    if n % 4 == 0:
        strings.append(bytes(fuzz.getrandbits(8) for _ in range(fuzz.randint(0, 40))))
        continue
    base = bytearray(bytes.fromhex(fuzz.choice(valid_rust)))
    action = fuzz.randrange(5)
    if action == 0 and base:
        base[fuzz.randrange(len(base))] ^= 1 << fuzz.randrange(8)
    elif action == 1:
        del base[fuzz.randint(0, len(base)):]
    elif action == 2 and base:
        del base[fuzz.randrange(len(base))]
    elif action == 3 and base:
        at = fuzz.randrange(len(base))
        base[at:at] = base[at:at + 1]
    else:
        base += bytes(fuzz.getrandbits(8) for _ in range(fuzz.randint(1, 8)))
    strings.append(bytes(base))
answers = batch(
    [{"id": i, "op": "decode", "form": "compact", "hex": s.hex()} for i, s in enumerate(strings)],
    "fuzz",
)
disagree = []
decoded = 0
for s, answer in zip(strings, answers):
    crate = {k: v for k, v in answer.items() if k in ("decoded", "error")}
    reference = python_decode(s.hex())
    decoded += "decoded" in crate
    if crate != reference:
        disagree.append((s.hex(), crate, reference))
DIFFS["fuzz"] = len(disagree)
print("byte strings:", len(strings), " decoded by both:", decoded, " refused by both:", len(strings) - decoded - len(disagree))
print("answers that differ:", len(disagree))
for s, crate, reference in disagree[:5]:
    print("  ", s[:80], crate, reference)

## Verdict

In [ ]:
print("differences by section:", DIFFS)
print("panics or crashes of the seam:", len(PANICS))
for panic in PANICS:
    print("  ", panic)
print("hostile packets refused with the expected error: %d of %d" % (refused, len(HOSTILE)))
print("header bytes of the ACK of one to five entries:",
      [HEADER_BYTES["ACK of %d" % n] for n in range(1, 6)])
print("header bytes of a part of DATA with an ACK and of LIMITS:",
      HEADER_BYTES["DATA part with ACK"], HEADER_BYTES["LIMITS"])
assert not PANICS, PANICS
assert sum(DIFFS.values()) == 0, DIFFS
assert refused == len(HOSTILE) == 64
print()
print("VERDICT: 0 byte differences over the 8 types and %d seeded valid packets, %d of %d hostile"
      " packets refused with the expected error, %d damaged strings read alike, 0 panics"
      % (len(valid), refused, len(HOSTILE), len(strings)))